## Demonstration of ogcapi API

The initial attempt to use the `owslib` library failed due to issus in the requests getting through to the F5.

Work from here down uses `requests` directly.

In [ ]:
import folium
import requests

### Get some GeoJSON data and display it directly on a `folium` map.

In [ ]:
# Get the data
params = {'f': 'json', 'limit': 1000}
response = requests.get('https://ogcapi.bgs.ac.uk/collections/offshore-sample-geochemical-data/items', params=params)
data = response.json()
data

In [ ]:
# Define an initial map centred on and zoomed to the UK
m = folium.Map(location=[55, -2], zoom_start=5)

In [ ]:
# Create a basic unstyled tooltip
tooltip = folium.features.GeoJsonTooltip(
    fields=['zn_ppm'],
    aliases=['Zinc ppm'])

In [ ]:
# Populate and display the map
folium.GeoJson(
    data,
    tooltip=tooltip,
    marker=folium.CircleMarker()
).add_to(m)
m

### Now get the data into geopandas, transform it and then create a map

In [ ]:
import geopandas as gpd
import pandas as pd
import branca.colormap as cm
from shapely.geometry import Polygon

In [ ]:
# Get all the data in batches of 'limit'.
# This is a bit ugly, there may be a neater way.
sample_list = []
offset = 0
limit = 500
params = {'f': 'json', 'offset': offset, 'limit': limit}
response = requests.get('https://ogcapi.bgs.ac.uk/collections/offshore-sample-geochemical-data/items', params=params)
data = response.json()
while len(data['features']) == limit:
    # Get JSON into GeoPandas
    sample_list.append(gpd.GeoDataFrame.from_features(data['features']))
    # Get the next batch
    offset += limit
    params = {'f': 'json', 'offset': offset, 'limit': limit}
    response = requests.get('https://ogcapi.bgs.ac.uk/collections/offshore-sample-geochemical-data/items', params=params)
    data = response.json()

# Add the final batch if not empty
if len(data['features']) > 0:
    sample_list.append(gpd.GeoDataFrame.from_features(data['features']))

# Concatenate dataframes into one
samples = pd.concat(
    sample_list,
    ignore_index=True,
)
# Set geometry
samples = samples.set_geometry('geometry')
samples.tail()

In [ ]:
samples.dtypes

In [ ]:
# Rename, convert and select columns. Set the CRS.
samples.rename(columns={'id': 'id', 'b_ppm ': 'b_ppm '}, inplace=True)
# samples['datetime'] = pd.to_datetime(samples['datetime']) # leave as string for now
samples['id'] = samples['id'].astype(int)
samples = samples[['id', 'client', 'water_depth', 'cu_ppm ', 'geometry']]
samples.set_crs(epsg=4326, inplace=True)
samples.head()

In [ ]:
samples.dtypes

In [ ]:
samples.describe()